# 03 · Night Shift (contest exercise)

**Style C: contest simulation** of T26-H-NIGHT ("Operation Night Watch", IOAI 2026 at-home): extend a
**deployed** classifier with new classes while a tiny, unbalanced remnant of the old data is all that is left.
It uses the same ½·acc_old + ½·acc_new metric, run on images instead of audio so it trains in minutes on a free GPU.
**Traces to** `notes/ioai-task-patterns.md` P1, P2, P4, P7.

## 1. Problem Description

A warehouse camera runs a CNN that recognises 5 garment types. Two new product lines add **5 new classes**.
Deploying a brand-new architecture is not possible; you must **start from the deployed weights**. The original
training data was deleted, apart from a small archive.

## 2. Dataset (FashionMNIST, 28×28 grey-scale)

| Data | Classes | Images |
|---|---|---|
| `deployed.pt` | trained on 0–4 | state dict of `Net(5)` |
| `X_old, y_old` (archive) | 0–4 | **12, 3, 20, 8, 25** per class |
| `X_new, y_new` | 5–9 | **40, 60, 24, 60, 45** per class |
| `X_test_a` / `X_test_b` | 0–9 | 5,000 each, labels hidden |

## 3. Task

Produce a 10-way classifier and predict every test image.

## 4. Submission

`submission.zip` containing `submission_a.npy` and `submission_b.npy`: 1-D **int64** arrays of length 5,000
with values 0–9.

## 5. Scoring

`0.5 · accuracy on old-class images + 0.5 · accuracy on new-class images`, computed per split.
A model that forgets everything old scores no better than one that learns nothing new.

- Baseline (Leaderboard B): **0.46** (grow the head, fine-tune on the new data only)
- Reference (Leaderboard B): **0.85**, which is your target

## 6. Requirements

- You must start from `deployed.pt` (no training a fresh network from scratch) and use only the data above.
- ≤ 5 minutes on a T4 for the whole notebook. torch/torchvision only, no pretrained models.

In [ ]:
import time, os, io, zipfile, random, copy
T0 = time.time()
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets
seed = 42
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
device = "cuda" if torch.cuda.is_available() else "cpu"

### Deployed model and data (DO NOT CHANGE)

In [ ]:
class Net(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(), nn.Linear(64 * 7 * 7, 128), nn.ReLU())
        self.head = nn.Linear(128, n_classes)
    def forward(self, x):
        return self.head(self.features(x))

def _setup():
    tr = datasets.FashionMNIST("data", train=True, download=True)
    X, y = tr.data.unsqueeze(1).float() / 255, tr.targets
    torch.manual_seed(2026)
    net = Net(5).to(device); opt = torch.optim.Adam(net.parameters(), 1e-3); m = y < 5
    for _ in range(2):
        for xb, yb in DataLoader(TensorDataset(X[m], y[m]), batch_size=64, shuffle=True):
            loss = F.cross_entropy(net(xb.to(device)), yb.to(device)); opt.zero_grad(); loss.backward(); opt.step()
    torch.save(net.state_dict(), "deployed.pt")
    g = torch.Generator().manual_seed(7)
    def take(counts):
        return torch.cat([torch.nonzero(y == c).flatten()[torch.randperm(int((y == c).sum()), generator=g)[:k]] for c, k in counts.items()])
    i_old = take({0: 12, 1: 3, 2: 20, 3: 8, 4: 25}); i_new = take({5: 40, 6: 60, 7: 24, 8: 60, 9: 45})
    return X[i_old], y[i_old], X[i_new], y[i_new]

X_old, y_old, X_new, y_new = _setup()
_te = datasets.FashionMNIST("data", train=False, download=True)
X_test = _te.data.unsqueeze(1).float() / 255
X_test_a, X_test_b = X_test[:5000], X_test[5000:]
del _te
print(X_old.shape, torch.bincount(y_old).tolist(), X_new.shape, torch.bincount(y_new)[5:].tolist())

### Baseline (replace this)

In [ ]:
def build_model():
    state = torch.load("deployed.pt", map_location=device)
    model = Net(10).to(device)
    model.load_state_dict({k: v for k, v in state.items() if not k.startswith("head.")}, strict=False)
    with torch.no_grad():
        model.head.weight[:5] = state["head.weight"]; model.head.bias[:5] = state["head.bias"]
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    for _ in range(3):
        model.train()
        for xb, yb in DataLoader(TensorDataset(X_new, y_new), batch_size=32, shuffle=True):
            loss = F.cross_entropy(model(xb.to(device)), yb.to(device))
            opt.zero_grad(); loss.backward(); opt.step()
    return model

### Train and validate locally

Old-class accuracy can be measured only on the tiny archive, so hold out a few images per class if you want an
honest local number. This quick check reuses the training data and is optimistic.

In [ ]:
model = build_model()

@torch.no_grad()
def predict(model, X):
    model.eval()
    return torch.cat([model(X[i:i + 1000].to(device)).argmax(1).cpu() for i in range(0, len(X), 1000)])

p_old, p_new = predict(model, X_old), predict(model, X_new)
print("train-set acc old:", (p_old == y_old).float().mean().item(), " new:", (p_new == y_new).float().mean().item())

### Submission

In [ ]:
def npy_bytes(a):
    buf = io.BytesIO(); np.save(buf, a); return buf.getvalue()
sub_a, sub_b = predict(model, X_test_a).numpy().astype(np.int64), predict(model, X_test_b).numpy().astype(np.int64)
with zipfile.ZipFile("submission.zip", "w") as z:
    z.writestr("submission_a.npy", npy_bytes(sub_a)); z.writestr("submission_b.npy", npy_bytes(sub_b))
print("wrote submission.zip", sub_a[:10])

### Official grader (run it, don't read it)

In [ ]:
SHOW_FINAL = False or os.environ.get("IOAI_SHOW_FINAL") == "1"

def _grade(zip_path="submission.zip"):
    y = datasets.FashionMNIST("data", train=False, download=True).targets.numpy()
    out = []
    with zipfile.ZipFile(zip_path) as z:
        for name, truth in [("submission_a.npy", y[:5000]), ("submission_b.npy", y[5000:])]:
            try:
                p = np.load(io.BytesIO(z.read(name)))
                assert p.shape == truth.shape and p.dtype.kind in "iu" and p.min() >= 0 and p.max() <= 9
                old, new = truth < 5, truth >= 5
                out.append(0.5 * (p[old] == truth[old]).mean() + 0.5 * (p[new] == truth[new]).mean())
            except Exception as e:
                print(name, "rejected:", e); out.append(0.0)
    return out

score_a, score_b = _grade()
print(f"Leaderboard A (public): {score_a:.4f}")
print(f"Leaderboard B (final):  {score_b:.4f}" if SHOW_FINAL else "Leaderboard B: hidden until SHOW_FINAL = True")
print(f"total runtime {(time.time() - T0) / 60:.1f} min (limit 5 min)")